# SEC dates and report metadata alignment

Adapted from the supplied original workflow. Outputs and personal paths have been removed. Read `docs/PREPARING_DATA.md` and configure `configs/preparation.local.json` first. Run cells in order. This adaptation has not been validated by a complete raw-to-neural rerun.


# Full-sample SEC filing-date workflow

This notebook supplements the competition-disclosure dataset with SEC filing dates. It reads **all completed, accepted reports in a frozen snapshot of the extraction manifest**, rather than the small pilot CSV or the first Excel part. It does not re-extract Item 1, recompute embeddings, calculate ROA, or run GENI.

Run steps 1–6 in order. This Windows version checks the full accepted manifest against the completed MiniLM sample when its snapshot is available. Financial matching is a separate next step. All code and comments are in English. CPU only. One notebook writer at a time per output folder.

**Scope:** “full sample” means every available report in the manifest at the time of this run. It does not imply that all 2010–2026 downloads or extractions have finished. Re-run after extraction grows. No fixed company count or pilot-report filter is used.

**Matching:** use CIK and the full accession number. Never infer the issuer CIK from the first ten digits of an accession, and never use company name or fiscal year to guess a filing date. Unmatched and ambiguous reports remain in the output.

**Resume:** raw JSON and completed company histories are cached. Restart the kernel and run steps 1–6 again after an interruption. An interrupted company is retried, while fresh completed company checkpoints are reused. Completed company checkpoints and main company JSON refresh after 24 hours by default. Historical JSON files are reused. To refresh now, set FORCE_REFRESH=True for one run and then set it back to False.

**Research use:** a verified submission date establishes disclosure timing, not causal identification. Next-fiscal-year ROA may include months before submission. Keep the current outcome specification until you explicitly choose and recompute an alternative. This notebook never changes outcomes automatically.

Sources: [SEC API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces), [SEC developer resources](https://www.sec.gov/about/developer-resources). Company JSON contains recent filings plus references to older filing-history files. The code follows those references for the selected filing-date window. It keeps requests sequential, identifies the researcher, and limits request frequency.


## Imports and local configuration

Set `SEC_CONTACT_EMAIL` to your own contact email before requesting public SEC metadata. Set the frozen extraction and existing embedding input paths in the preparation configuration.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import defaultdict
import hashlib
import json
import os
import re
import time
import uuid

import pandas as pd
import requests
from tqdm.auto import tqdm

from pathlib import Path
import sys
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from preparation_config import load_preparation_config
PREP = load_preparation_config(REPO_ROOT)

INPUT_ROOT = PREP["extraction_dir"]
DATA_DIR = INPUT_ROOT / "all_years" if (INPUT_ROOT / "all_years").is_dir() else INPUT_ROOT
MANIFEST_PATH = DATA_DIR / "cache_manifest.jsonl"
OUTPUT_ROOT = PREP["sec_output_root"]
JSON_CACHE = OUTPUT_ROOT / "json_cache"
COMPANY_CACHE = OUTPUT_ROOT / "company_checkpoints"

# Current completed MiniLM sample. Update these only when the sample changes.
EXPECTED_REPORTS = 106877
EXPECTED_CORPUS_ID = "fa8fc124c14a1ca97e34"
MINILM_ROOT = PREP["embedding_root"] / "minilm" / EXPECTED_CORPUS_ID

SEC_EMAIL = os.environ.get("SEC_CONTACT_EMAIL", "")
START_DATE = "2010-01-01"
END_DATE = "2026-12-31"
FORMS = ["10-K", "10-K/A"]
MAX_COMPANIES = None
CACHE_TTL_HOURS = 24
FORCE_REFRESH = False
REQUEST_INTERVAL_SECONDS = 0.35
CODE_VERSION = "sec_filing_dates_windows_v2"

assert MANIFEST_PATH.is_file(), f"Missing extraction manifest: {MANIFEST_PATH}"
if not SEC_EMAIL:
    SEC_EMAIL = input("Enter your real contact email for SEC requests: ").strip()
SEC_EMAIL = SEC_EMAIL.strip()
assert "@" in SEC_EMAIL and SEC_EMAIL.isascii() and not any(c.isspace() for c in SEC_EMAIL), "Enter a valid ASCII contact email without spaces."
assert pd.Timestamp(START_DATE) <= pd.Timestamp(END_DATE)
assert MAX_COMPANIES is None or MAX_COMPANIES > 0

for folder in [OUTPUT_ROOT, JSON_CACHE, COMPANY_CACHE]:
    folder.mkdir(parents=True, exist_ok=True)

POLICY = {"version": CODE_VERSION, "start": START_DATE, "end": END_DATE, "forms": sorted(FORMS)}
POLICY_ID = hashlib.sha256(json.dumps(POLICY, sort_keys=True).encode()).hexdigest()[:16]
COMPANY_CACHE = COMPANY_CACHE / POLICY_ID
COMPANY_CACHE.mkdir(parents=True, exist_ok=True)

print("Manifest:", MANIFEST_PATH)
print("Scope:", "ALL manifest companies" if MAX_COMPANIES is None else f"TRIAL: {MAX_COMPANIES} companies")
print("Outputs:", OUTPUT_ROOT)


## 2. Freeze the manifest and define identifier helpers

The snapshot has a fixed byte length even if extraction continues writing to the original manifest. Only an incomplete final JSON line without a newline can be skipped. Other malformed lines raise an error. Original files remain unchanged.


In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def atomic_json(path, value):
    path = Path(path)
    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    os.replace(temp, path)

def atomic_csv(frame, path):
    path = Path(path)
    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    frame.to_csv(temp, index=False, compression="gzip" if path.name.endswith(".gz") else None)
    os.replace(temp, path)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def clean_cik(value):
    if value is None or pd.isna(value):
        return None
    value = re.sub(r"\.0$", "", str(value).strip())
    if not re.fullmatch(r"\d{1,10}", value) or int(value) == 0:
        return None
    return value.zfill(10)

def accession_candidates(value):
    if value is None or pd.isna(value):
        return set()
    text = str(value)
    found = set(re.findall(r"(?<!\d)(\d{10}-\d{2}-\d{6})(?!\d)", text))
    for x in re.findall(r"(?<!\d)(\d{18})(?!\d)", text):
        found.add(f"{x[:10]}-{x[10:12]}-{x[12:]}")
    return found

def source_filename_cik(source):
    # Lean16 filename convention: issuerCIK_accession_document.html.
    # The accession prefix can belong to a filing agent, so do not use it as CIK.
    name = PurePosixPath(str(source).replace("\\", "/").split("!")[-1]).name
    match = re.match(r"^(\d{1,10})_\d{10}-\d{2}-\d{6}_", name)
    return clean_cik(match.group(1)) if match else None

RUN_DIR = OUTPUT_ROOT / "runs" / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8]
)
RUN_DIR.mkdir(parents=True, exist_ok=False)
MANIFEST_SNAPSHOT = RUN_DIR / "source_manifest.jsonl"
initial_bytes = MANIFEST_PATH.stat().st_size
remaining = initial_bytes
with MANIFEST_PATH.open("rb") as src, MANIFEST_SNAPSHOT.open("wb") as dst:
    while remaining:
        chunk = src.read(min(1024 * 1024, remaining))
        if not chunk:
            raise RuntimeError("Manifest changed or shrank during snapshot. Rerun from step 2.")
        dst.write(chunk)
        remaining -= len(chunk)

MANIFEST_SHA256 = sha256_file(MANIFEST_SNAPSHOT)
print("Frozen manifest bytes:", initial_bytes)
print("Run folder:", RUN_DIR)


## 3. Index every completed report

This uses identifiers in the manifest and accession numbers in source paths. If a required identifier is absent, the report remains unresolved. Conflicting manifest records remain in the output with a blocking flag. This step does not reopen or parse the full HTML collection. An “accepted” extraction status does not certify Item 1 boundary accuracy.


In [ ]:
def index_manifest(path):
    by_doc = {}
    stats = {"manifest_lines": 0, "accepted_entries": 0, "ignored_entries": 0,
             "incomplete_final_lines": 0, "accepted_entries_without_doc_id": 0}
    with Path(path).open("rb") as handle:
        for line_number, raw in enumerate(handle, 1):
            if not raw.strip():
                continue
            stats["manifest_lines"] += 1
            try:
                record = json.loads(raw)
            except (ValueError, UnicodeDecodeError):
                if not raw.endswith(b"\n"):
                    stats["incomplete_final_lines"] += 1
                    continue
                raise ValueError(f"Malformed manifest JSON at line {line_number}")
            status = record.get("status", record.get("extract_status"))
            if status != "ok" or record.get("duplicate") is True:
                stats["ignored_entries"] += 1
                continue
            doc_id = str(record.get("doc_id") or "").strip()
            if not doc_id:
                stats["accepted_entries_without_doc_id"] += 1
                raise ValueError(f"Accepted entry has no doc_id at line {line_number}")
            stats["accepted_entries"] += 1
            entry = by_doc.setdefault(doc_id, {"ciks": set(), "accessions": set(),
                                             "sources": set(), "n_entries": 0})
            entry["n_entries"] += 1
            source = str(record.get("source_path") or "")
            if source:
                entry["sources"].add(source)
            for value in [record.get("cik"), source_filename_cik(source)]:
                cik = clean_cik(value)
                if cik:
                    entry["ciks"].add(cik)
            for key in ["accession", "accession_number", "accessionNumber", "source_path"]:
                entry["accessions"].update(accession_candidates(record.get(key)))

    rows = []
    for doc_id, entry in by_doc.items():
        nc, na = len(entry["ciks"]), len(entry["accessions"])
        if nc == 0:
            status = "cik_missing"
        elif nc > 1:
            status = "conflicting_cik_candidates"
        elif na == 0:
            status = "accession_missing"
        elif na > 1:
            status = "conflicting_accession_candidates"
        else:
            status = "identifier_ready"
        rows.append({
            "doc_id": doc_id,
            "cik": next(iter(entry["ciks"])) if nc == 1 else None,
            "accession": next(iter(entry["accessions"])) if na == 1 else None,
            "identifier_status": status,
            "n_cik_candidates": nc,
            "n_accession_candidates": na,
            "n_manifest_entries": entry["n_entries"],
            "source_path": sorted(entry["sources"])[0] if entry["sources"] else "",
            "candidate_ciks": ";".join(sorted(entry["ciks"])),
            "candidate_accessions": ";".join(sorted(entry["accessions"])),
        })
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise ValueError("No accepted reports in the manifest snapshot.")
    return frame.sort_values("doc_id").reset_index(drop=True), stats

report_index, manifest_stats = index_manifest(MANIFEST_SNAPSHOT)
all_ciks = sorted(report_index["cik"].dropna().unique().tolist())
selected_ciks = all_ciks if MAX_COMPANIES is None else all_ciks[:MAX_COMPANIES]
assert selected_ciks, "No valid CIK found in this manifest."
atomic_csv(report_index, RUN_DIR / "report_identifier_index.csv.gz")
atomic_json(RUN_DIR / "manifest_inventory.json", {
    **manifest_stats, "reports": len(report_index), "companies": len(all_ciks),
    "selected_companies": len(selected_ciks), "manifest_sha256": MANIFEST_SHA256,
    "policy": POLICY,
})
print("All accepted reports in snapshot:", len(report_index))
print("Valid company CIKs:", len(all_ciks))
print("Companies selected for download:", len(selected_ciks))
print("\nIdentifier status:")
print(report_index["identifier_status"].value_counts())
print("\nIncomplete final manifest line:", manifest_stats["incomplete_final_lines"])
display(report_index.head())


# Verify the full-sample scope before any SEC network requests.
if EXPECTED_REPORTS is not None:
    assert len(report_index) == EXPECTED_REPORTS, (
        f"Expected {EXPECTED_REPORTS:,} accepted reports, found {len(report_index):,}. Check MANIFEST_PATH."
    )
minilm_alignment = {"status": "minilm_snapshot_not_available", "expected_corpus_id": EXPECTED_CORPUS_ID}
snapshots = list(MINILM_ROOT.glob("*/latest_snapshot.json")) if MINILM_ROOT.is_dir() else []
if snapshots:
    snapshot_path = max(snapshots, key=lambda p:p.stat().st_mtime_ns)
    snapshot = json.loads(snapshot_path.read_text(encoding="utf-8"))
    assert snapshot.get("complete") is True and snapshot.get("corpus_id") == EXPECTED_CORPUS_ID
    assert snapshot.get("settings",{}).get("model") == "sentence-transformers/all-MiniLM-L6-v2"
    mini_index_path = snapshot_path.parent / snapshot["report_index"]
    assert mini_index_path.is_file(), f"Missing MiniLM index: {mini_index_path}"
    assert sha256_file(mini_index_path) == snapshot["report_index_sha256"], "MiniLM index checksum differs."
    mini_index = pd.read_csv(mini_index_path, dtype={"doc_id":"string","cik":"string"})
    assert mini_index["doc_id"].notna().all() and mini_index["doc_id"].is_unique
    assert set(mini_index["doc_id"]) == set(report_index["doc_id"]), "SEC and MiniLM report IDs differ."
    compared = report_index[["doc_id","cik"]].merge(mini_index[["doc_id","cik"]],
        on="doc_id",validate="one_to_one",suffixes=("_sec","_minilm"))
    a=compared["cik_sec"].map(clean_cik).astype("string")
    b=compared["cik_minilm"].map(clean_cik).astype("string")
    conflicts=a.notna() & b.notna() & a.ne(b).fillna(False)
    assert not conflicts.any(), "CIK conflicts between the manifest and MiniLM index require review."
    minilm_alignment={"status":"exact_doc_id_set_verified", "corpus_id":EXPECTED_CORPUS_ID,
        "reports":len(mini_index),"index":str(mini_index_path),"index_sha256":snapshot["report_index_sha256"]}
    print("MiniLM alignment: PASS — identical report IDs; comparable CIKs agree.")
else:
    print("MiniLM snapshot not found locally. Expected report count checked; exact MiniLM alignment not verified.")
atomic_json(RUN_DIR / "minilm_scope_check.json", minilm_alignment)
print("Scope checks passed. Continue with Steps 4–6.")


## 4. Define the cached SEC downloader

An HTTP 403 or 429 stops this run to avoid repeatedly requesting after an access block. Other failures are recorded for retry on the next run. Four attempts are allowed for transient connection/server failures. The code never stores a failed or incomplete company history as successful. Do not run multiple SEC download notebooks simultaneously.

Existing JSON from the earlier pilot is reused if it is still fresh. Each complete company checkpoint stores the requested date window and form policy. No API call is needed to reuse a fresh company checkpoint.


In [ ]:
BASE_URL = "https://data.sec.gov/submissions/"
SEC_COLUMNS = ["accessionNumber", "filingDate", "reportDate", "acceptanceDateTime", "form", "primaryDocument"]
OUTPUT_COLUMNS = ["cik", "accession", "sec_filing_date", "sec_report_period_end",
                  "sec_acceptance_datetime", "sec_form", "sec_primary_document"]
session = requests.Session()
session.headers.update({
    "User-Agent": f"Academic research {SEC_EMAIL}",
    "Accept": "application/json",
    "Accept-Encoding": "gzip, deflate",
})
last_request_started = 0.0
network_requests = 0
json_cache_hits = 0

class SECAccessBlocked(RuntimeError):
    pass

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def is_fresh(path):
    return Path(path).is_file() and (time.time() - Path(path).stat().st_mtime) < CACHE_TTL_HOURS * 3600

def get_sec_json(filename, recent=False):
    global last_request_started, network_requests, json_cache_hits
    if PurePosixPath(filename).name != filename or not filename.endswith(".json"):
        raise ValueError(f"Unexpected SEC JSON filename: {filename}")
    path = JSON_CACHE / filename
    if not FORCE_REFRESH and path.is_file() and (not recent or is_fresh(path)):
        try:
            value = read_json(path)
            if isinstance(value, dict):
                json_cache_hits += 1
                return value
        except (ValueError, OSError):
            pass

    for attempt in range(4):
        wait = REQUEST_INTERVAL_SECONDS - (time.monotonic() - last_request_started)
        if wait > 0:
            time.sleep(wait)
        last_request_started = time.monotonic()
        network_requests += 1
        try:
            response = session.get(BASE_URL + filename, timeout=(15, 60))
            if response.status_code in [403, 429]:
                raise SECAccessBlocked(
                    f"SEC returned HTTP {response.status_code}. Run stopped. "
                    "Check the contact email and network, and retry later."
                )
            response.raise_for_status()
            data = response.json()
            if not isinstance(data, dict):
                raise ValueError("SEC JSON must be an object")
            atomic_json(path, data)
            return data
        except SECAccessBlocked:
            raise
        except requests.HTTPError as exc:
            if response.status_code < 500 or attempt == 3:
                raise
        except (requests.RequestException, ValueError):
            if attempt == 3:
                raise
        time.sleep(2 ** (attempt + 1))

def annual_rows(block, cik):
    if not isinstance(block, dict) or "accessionNumber" not in block:
        raise ValueError("Unrecognized SEC filing-history schema")
    n = len(block["accessionNumber"])
    data = {}
    for col in SEC_COLUMNS:
        values = block.get(col, [None] * n)
        if not isinstance(values, list) or len(values) != n:
            raise ValueError(f"Invalid SEC array length: {col}")
        data[col] = values
    frame = pd.DataFrame(data)
    dates = pd.to_datetime(frame["filingDate"], errors="coerce")
    frame = frame.loc[frame["form"].isin(FORMS) & dates.between(START_DATE, END_DATE)].copy()
    frame.insert(0, "cik", cik)
    frame = frame.rename(columns={
        "accessionNumber": "accession", "filingDate": "sec_filing_date",
        "reportDate": "sec_report_period_end", "acceptanceDateTime": "sec_acceptance_datetime",
        "form": "sec_form", "primaryDocument": "sec_primary_document",
    })
    return frame[OUTPUT_COLUMNS]

def company_checkpoint_path(cik):
    return COMPANY_CACHE / f"CIK{cik}.json"

def load_company(cik):
    path = company_checkpoint_path(cik)
    if not FORCE_REFRESH and is_fresh(path):
        try:
            checkpoint = read_json(path)
            if checkpoint.get("policy") == POLICY and checkpoint.get("cik") == cik:
                return checkpoint, "company_checkpoint"
        except (ValueError, OSError):
            pass

    main = get_sec_json(f"CIK{cik}.json", recent=True)
    if clean_cik(main.get("cik")) != cik:
        raise ValueError("CIK in SEC response differs from requested CIK")
    filings = main.get("filings")
    if not isinstance(filings, dict) or "recent" not in filings:
        raise ValueError("Main SEC response has no filings.recent")
    frames = [annual_rows(filings["recent"], cik)]
    used_files = [f"CIK{cik}.json"]

    for item in filings.get("files", []):
        filing_from, filing_to = item.get("filingFrom", ""), item.get("filingTo", "")
        if filing_to and filing_to < START_DATE:
            continue
        if filing_from and filing_from > END_DATE:
            continue
        filename = item["name"]
        frames.append(annual_rows(get_sec_json(filename), cik))
        used_files.append(filename)

    frame = pd.concat(frames, ignore_index=True).drop_duplicates()
    if frame.duplicated(["cik", "accession"]).any():
        raise ValueError("Conflicting SEC records for the same CIK and accession")
    if frame["accession"].isna().any():
        raise ValueError("Annual SEC record has no accession")
    checkpoint = {
        "cik": cik, "policy": POLICY, "checked_at_utc": utc_now(),
        "sec_current_company_name": main.get("name"),
        "files": used_files,
        "records": frame.sort_values(["sec_filing_date", "accession"]).to_dict("records"),
    }
    atomic_json(path, checkpoint)
    return checkpoint, "download_or_json_cache"


## 5. Download every selected company, with checkpoints

You can interrupt this cell. Run steps 1–6 again to continue later. A newly created run folder preserves the earlier run's outputs. All runs share the JSON and company checkpoint caches. The progress bar includes cached companies, which can finish very quickly.

If some companies fail or the run stops, step 6 can still export a clearly marked partial result. Unattempted companies remain visible. Do not label such a result fully downloaded.


In [ ]:
download_status = {
    cik: {"cik": cik, "download_status": "not_attempted", "origin": "", "annual_records": None, "error": ""}
    for cik in selected_ciks
}
try:
    for cik in tqdm(selected_ciks, desc="SEC metadata: all selected companies"):
        try:
            checkpoint, origin = load_company(cik)
            download_status[cik].update(
                download_status="ok", origin=origin, annual_records=len(checkpoint["records"])
            )
        except SECAccessBlocked as exc:
            download_status[cik].update(download_status="access_blocked", error=str(exc))
            print(str(exc))
            break
        except KeyboardInterrupt:
            download_status[cik].update(download_status="interrupted", error="Interrupted by user")
            print("Interrupted. Completed company checkpoints are saved.")
            break
        except Exception as exc:
            download_status[cik].update(download_status="failed", error=f"{type(exc).__name__}: {exc}")
finally:
    download_status_df = pd.DataFrame(download_status.values())
    atomic_csv(download_status_df, RUN_DIR / "company_download_status.csv")

print(download_status_df["download_status"].value_counts())
print("Network requests:", network_requests, "| Raw JSON cache hits:", json_cache_hits)
print("Saved:", RUN_DIR / "company_download_status.csv")


## 6. Align exact reports and export the full audit table

The output has one row per manifest doc_id, including unresolved reports. SEC dates come only from an exact CIK–accession match, with the selected annual form type. Multiple text documents can legitimately refer to the same SEC submission, so the output flags repeated submissions rather than deleting them.

The date table does not certify that every report has a valid financial match, a complete Item 1, or a usable future outcome. Those remain separate checks in the financial/GENI workflow.


In [ ]:
def build_metadata(status_frame):
    rows = []
    for cik in status_frame.loc[status_frame["download_status"].eq("ok"), "cik"]:
        checkpoint = read_json(company_checkpoint_path(cik))
        if checkpoint.get("policy") != POLICY or checkpoint.get("cik") != cik:
            raise ValueError(f"Company checkpoint mismatch: {cik}")
        rows.extend(checkpoint["records"])
    frame = pd.DataFrame(rows, columns=OUTPUT_COLUMNS)
    frame["cik"] = frame["cik"].astype("string")
    frame["accession"] = frame["accession"].astype("string")
    assert not frame.duplicated(["cik", "accession"]).any()
    return frame

def align_reports(index, metadata, status_frame):
    left = index.copy()
    left["cik"] = left["cik"].astype("string")
    left["accession"] = left["accession"].astype("string")
    # Block every ambiguous identifier even if a value happens to be present.
    ready = left["identifier_status"].eq("identifier_ready")
    left["_match_cik"] = left["cik"].where(ready)
    left["_match_accession"] = left["accession"].where(ready)
    right = metadata.rename(columns={"cik": "_match_cik", "accession": "_match_accession"})
    right = right.loc[right["_match_cik"].notna() & right["_match_accession"].notna()]
    result = left.merge(right, on=["_match_cik", "_match_accession"], how="left", validate="many_to_one")
    result = result.drop(columns=["_match_cik", "_match_accession"])
    result = result.merge(status_frame[["cik", "download_status"]], on="cik", how="left", validate="many_to_one")
    result["filing_date_match_status"] = result["identifier_status"]
    result.loc[ready.to_numpy(), "filing_date_match_status"] = "no_exact_sec_match_in_window"
    result.loc[ready.to_numpy() & result["download_status"].isna(), "filing_date_match_status"] = "company_not_selected_for_trial"
    result.loc[ready.to_numpy() & result["download_status"].notna() & result["download_status"].ne("ok"),
               "filing_date_match_status"] = "company_metadata_unavailable"
    matched = result["sec_filing_date"].notna()
    result.loc[matched, "filing_date_match_status"] = "matched_cik_and_accession"
    result["filing_date_verified"] = matched
    result["filing_date_source"] = pd.NA
    result.loc[matched, "filing_date_source"] = "SEC_submissions_exact_cik_accession"
    filing = pd.to_datetime(result["sec_filing_date"], errors="coerce")
    period = pd.to_datetime(result["sec_report_period_end"], errors="coerce")
    result["days_period_end_to_filing"] = (filing - period).dt.days
    result["date_order_review_required"] = filing.notna() & period.notna() & filing.lt(period)
    usable_key = ready.to_numpy()
    result["same_submission_multiple_docs"] = usable_key & result.duplicated(["cik", "accession"], keep=False)
    assert len(result) == len(index) and result["doc_id"].is_unique
    return result

metadata = build_metadata(download_status_df)
date_table = align_reports(report_index, metadata, download_status_df)
METADATA_PATH = RUN_DIR / "sec_annual_filing_metadata.csv.gz"
DATE_TABLE_PATH = RUN_DIR / "report_filing_dates_all.csv.gz"
atomic_csv(metadata, METADATA_PATH)
atomic_csv(date_table, DATE_TABLE_PATH)
atomic_csv(date_table.loc[
    ~date_table["filing_date_verified"] | date_table["date_order_review_required"] | date_table["same_submission_multiple_docs"]
], RUN_DIR / "filing_date_review.csv.gz")

summary = {
    "created_at_utc": utc_now(), "code_version": CODE_VERSION,
    "minilm_alignment": minilm_alignment,
    "manifest_sha256": MANIFEST_SHA256, "policy": POLICY,
    "reports_in_manifest": len(report_index), "valid_company_ciks": len(all_ciks),
    "selected_companies": len(selected_ciks),
    "successful_companies": int(download_status_df["download_status"].eq("ok").sum()),
    "download_status_counts": download_status_df["download_status"].value_counts().to_dict(),
    "all_manifest_companies_downloaded": MAX_COMPANIES is None and bool(download_status_df["download_status"].eq("ok").all()),
    "reports_without_unique_cik": int(report_index["cik"].isna().sum()),
    "all_reports_matched": bool(date_table["filing_date_verified"].all()),
    "filing_date_match_counts": date_table["filing_date_match_status"].value_counts().to_dict(),
    "incomplete_manifest_final_line": manifest_stats["incomplete_final_lines"],
    "date_order_review_reports": int(date_table["date_order_review_required"].sum()),
    "same_submission_multiple_docs": int(date_table["same_submission_multiple_docs"].sum()),
    "files": {"metadata": str(METADATA_PATH), "report_dates": str(DATE_TABLE_PATH)},
    "report_dates_sha256": sha256_file(DATE_TABLE_PATH),
    "scope_note": "All accepted manifest reports at the frozen snapshot time, not proof of collection completeness.",
}
atomic_json(RUN_DIR / "run_summary.json", summary)
atomic_json(OUTPUT_ROOT / "latest_run.json", {"run_dir": str(RUN_DIR), **summary})

print("Reports:", len(date_table))
print("Annual SEC filing records:", len(metadata))
print("All valid manifest companies downloaded:", summary["all_manifest_companies_downloaded"])
print("All reports matched:", summary["all_reports_matched"])
print("\nAlignment status:")
print(date_table["filing_date_match_status"].value_counts())
print("\nMain output:", DATE_TABLE_PATH)
display(date_table[["doc_id", "cik", "accession", "sec_form", "sec_report_period_end",
                    "sec_filing_date", "filing_date_match_status"]].head(15))


In [ ]:
from pathlib import Path
import json
import pandas as pd

# Locate the latest SEC export.
SEC_ROOT = PREP["sec_output_root"]
LATEST_FILE = SEC_ROOT / "latest_run.json"

assert LATEST_FILE.is_file(), f"Missing: {LATEST_FILE}"

latest = json.loads(LATEST_FILE.read_text(encoding="utf-8"))
RUN_DIR = Path(latest["run_dir"])

DATES_FILE = RUN_DIR / "report_filing_dates_all.csv.gz"
STATUS_FILE = RUN_DIR / "company_download_status.csv"

assert DATES_FILE.is_file(), f"Missing: {DATES_FILE}"
assert STATUS_FILE.is_file(), f"Missing: {STATUS_FILE}"

dates = pd.read_csv(
    DATES_FILE,
    dtype={"doc_id": "string", "cik": "string", "accession": "string"},
    low_memory=False,
)
companies = pd.read_csv(STATUS_FILE, dtype={"cik": "string"})

required = {
    "doc_id", "cik", "accession", "filing_date_match_status",
    "sec_filing_date", "sec_report_period_end", "sec_form",
}
assert required.issubset(dates.columns), (
    f"Missing columns: {sorted(required - set(dates.columns))}"
)

filing_date = pd.to_datetime(dates["sec_filing_date"], errors="coerce")
period_end = pd.to_datetime(dates["sec_report_period_end"], errors="coerce")

exact_match = dates["filing_date_match_status"].eq(
    "matched_cik_and_accession"
)
matched = exact_match & filing_date.notna()
unmatched = ~matched

print("SEC output folder:", RUN_DIR)
print("\n===== REPORT MATCHING =====")
print(f"Total reports: {len(dates):,}")
print(f"Matched with a valid filing date: {matched.sum():,}")
print(f"Unmatched / unresolved: {unmatched.sum():,}")
print(f"Match rate: {matched.mean():.2%}")

print("\nMatching status:")
display(
    dates["filing_date_match_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="reports")
)

print("\n===== COMPANY DOWNLOADS =====")
display(
    companies["download_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="companies")
)

failed_companies = companies.loc[
    companies["download_status"].ne("ok")
]
if not failed_companies.empty:
    print("Companies requiring download review — first 20:")
    display(failed_companies.head(20))

print("\n===== COMPLETENESS AND DATE CHECKS =====")
checks = pd.DataFrame([
    {
        "check": "Report count equals current full sample (106,877)",
        "passed": len(dates) == 106877,
    },
    {
        "check": "Report IDs are nonmissing and unique",
        "passed": dates["doc_id"].notna().all()
                  and dates["doc_id"].is_unique,
    },
    {
        "check": "All selected company downloads succeeded",
        "passed": not companies.empty
                  and companies["download_status"].eq("ok").all(),
    },
    {
        "check": "Every exact match has a parseable filing date",
        "passed": not (exact_match & filing_date.isna()).any(),
    },
])
display(checks)

date_order_problem = (
    matched & period_end.notna() & filing_date.lt(period_end)
)
print("Matched reports missing a report-period end:",
      int((matched & period_end.isna()).sum()))
print("Filing date earlier than report-period end:",
      int(date_order_problem.sum()))

print("\n===== REPEATED SEC SUBMISSIONS =====")
# Count only exact matched submissions with complete keys.
submission_rows = dates.loc[
    matched & dates[["cik", "accession"]].notna().all(axis=1)
]
submission_counts = (
    submission_rows.groupby(["cik", "accession"])
    .size()
    .rename("report_documents")
    .reset_index()
)
repeated = submission_counts.loc[
    submission_counts["report_documents"].gt(1)
]

print("Repeated doc_id rows:",
      int(dates["doc_id"].duplicated(keep=False).sum()))
print("SEC submissions linked to multiple report documents:",
      len(repeated))
print("Report documents involved:",
      int(repeated["report_documents"].sum()))

if not repeated.empty:
    display(
        repeated.sort_values(
            "report_documents", ascending=False
        ).head(20)
    )

print("\n===== UNMATCHED REPORT EXAMPLES =====")
columns = [
    "doc_id", "cik", "accession", "filing_date_match_status"
]
if "source_path" in dates.columns:
    columns.append("source_path")

display(dates.loc[unmatched, columns].head(20))

print("\n===== MATCHED FORM TYPES =====")
display(dates.loc[matched, "sec_form"].value_counts(dropna=False))

print("\nRead-only inspection finished. No files were changed.")

## Reading the results and running again

| File | Purpose |
|---|---|
| `report_filing_dates_all.csv.gz` | Main result, one row per accepted manifest report, including failures |
| `sec_annual_filing_metadata.csv.gz` | SEC submission histories for successfully processed selected companies |
| `filing_date_review.csv.gz` | Missing identifiers, failed downloads, unmatched submissions or review flags |
| `company_download_status.csv` | Successful, failed, blocked and unattempted companies |
| `run_summary.json` | Scope, counts, date window and input fingerprint |
| `latest_run.json` in OUTPUT_ROOT | Location and summary of the most recent export |

`matched_cik_and_accession` confirms an exact submission match. `no_exact_sec_match_in_window` needs review of identifiers, date range and form type. A successful download with zero annual records is not the same as a failed download. Trial exclusions and unfinished downloads remain explicit.

To expand: finish or extend extraction, keep MAX_COMPANIES=None, then run steps 1–6 again. New companies and historical files download as needed. Existing company checkpoints are reusable until the refresh interval expires. For newly published filings, refresh the caches with FORCE_REFRESH=True for one run. For thousands of companies, SEC also offers an official bulk submissions ZIP as an alternative download strategy, but this notebook uses targeted per-company retrieval with checkpoints.

After dates align, assess the financial outcome window and company identity separately. Dates alone do not resolve Item 1 boundary errors, dictionary measurement, historical CIK–GVKEY ambiguity, confounding or causal identification. Do not automatically delete a report merely because the default next-fiscal-year ROA period overlaps the filing date. Choose a research window explicitly, then update outcomes in the financial workflow.


**Next step:** provide `report_filing_dates_all.csv.gz`, `company_download_status.csv` and `run_summary.json` for review. Then link financial records using the verified company/period key. Do not merge the pilot financial table into this full dataset here. To resume after interruption, run Steps 1–6 again with the same OUTPUT_ROOT and FORCE_REFRESH=False. A new run folder is created; shared completed checkpoints are reused subject to the 24-hour freshness policy.
